# MRIO to SRIO

This notebook transforms a Multi-Regional table into a Single-Regional table (MRIO-to-SRIO), working with a test IOT.
The logic behind this transformation is represented in the figure below: after selecting the reference region of the SRIO (Reg1 in this case), MARIO will aggregate all imports of such region into a new factor of production and all the exports in a new consumption category.

![shock_template](../../_static/images/MRIO_to_SRIO.png)

In [1]:
import mario

In [2]:
db = mario.load_test("IOT")

INFO Parser: excel reading IOT flows from /Users/lorenzorinaldi/Documents/GitHub/MARIO/mario/test/tables/test_IOT_standard.xlsx.
INFO Parser: state payload ready with 6 canonical blocks.
INFO Parser: excel state ready for IOT.
INFO Metadata: initialized.


## Inspect the starting regions

In [3]:
db.regions

['Reg1', 'Reg2']

In [4]:
db.factors

['Taxes', 'Wages', 'Capital']

In [5]:
db.consumption_categories

['Final demand']

## Keep only `Reg1`

In [6]:
single_region_db = db.to_single_region("Reg1", inplace=False)
single_region_db.regions

WARNING All the scenarios will be deleted to build up the new baseline.


INFO Resolver: resolving X for baseline.


INFO Resolver: trying X via formula build_iot_X_from_Z_Y.


INFO Resolver: resolved X via formula build_iot_X_from_Z_Y.


INFO Transformation: baseline deleted from the database.


INFO Transformation: New baseline added to the database


['Reg1']

## Keep more than one region

Use `to_region_subset(...)` to retain selected regions as endogenous nodes and externalize the rest.

The retained regions must form a strict subset. The packaged IOT has two regions, so this executable example retains one; a larger MRIO can retain several.


In [ ]:
regions_to_keep = db.regions[:-1]
subset_db = db.to_region_subset(regions_to_keep, inplace=False)
subset_db.regions

## Inspect the new value-added and final-demand labels

In [7]:
single_region_db.factors

['Taxes', 'Wages', 'Capital', 'imports']

In [8]:
single_region_db.consumption_categories

['Final demand', 'Final Demand exports', 'Intermediate exports']

## SUT imports by commodity or producing activity

The imported classification is a declared row dimension beside the single factor label `imports`. Original factors keep empty classification levels. `activity` refers to the foreign producer, using source supply shares. `trade_mode="aggregate"` sums foreign regions; `"by_region"` retains source regions as an explicit level. Direct final-demand imports remain in `VY`; domestic environmental accounts remain in `Ea`, `Ec` and `EY`.


In [ ]:
sut = mario.load_test("SUT")
region = sut.regions[0]
commodities = sut.to_single_region(region, imports_by="commodity", inplace=False)
producers = sut.to_single_region(region, imports_by="activity", inplace=False)
commodities.query(["S", "U", "Ya", "Yc", "Va", "Vc", "VY", "Ea", "Ec", "EY"])

In [ ]:
partner_commodities = sut.to_region_subset(
    [region], imports_by="commodity", trade_mode="by_region", inplace=False,
)
partner_commodities.factors

## Aggregate the classification once

This example combines the two commodity labels of the packaged SUT. The mapping also aggregates the Commodity row dimension on imports in `Va`, `Vc` and `VY`; no Factor of production mapping is needed.

In [ ]:
import pandas as pd
items = commodities.get_index("Commodity")
mapping = pd.DataFrame({"Aggregation": items}, index=items)
mapping.loc[items[:2], "Aggregation"] = "Combined commodities"
commodities.aggregate({"Commodity": mapping}, levels=["Commodity"], zero_output_epsilon=None)
commodities.query("VY").index

## IOT imports by sector

Sector is also a declared row dimension in `V` and `VY`. Select the imported classification independently of the retained source-region detail.

In [ ]:
sector_imports = db.to_single_region(
    db.regions[0], imports_by="sector", trade_mode="aggregate", inplace=False,
)
sector_imports.V.index.names